Implementing GPT-2 with TF

In [3]:
import tensorflow as tf
import os
import requests

In [ ]:
# Download novels from Project Gutenberg
DATASOURCE = {
    "moby_dick": "https://www.gutenberg.org/ebooks/2701.txt.utf-8",
    "frankenstein": "https://www.gutenberg.org/ebooks/84.txt.utf-8",
    "dracula": "https://www.gutenberg.org/ebooks/345.txt.utf-8",
    "little_women": "https://www.gutenberg.org/ebooks/37106.txt.utf-8",
    "pride_and_prejudice": "https://www.gutenberg.org/ebooks/1342.txt.utf-8",
    "alice_in_wonderland": "https://www.gutenberg.org/ebooks/11.txt.utf-8",
    "crime_and_punishment": "https://www.gutenberg.org/ebooks/2554.txt.utf-8",
    "tom_sawyer": "https://www.gutenberg.org/ebooks/74.txt.utf-8",
    "tale_of_two_cities": "https://www.gutenberg.org/ebooks/98.txt.utf-8",
    "sherlock_holmes": "https://www.gutenberg.org/ebooks/1661.txt.utf-8",
    "war_and_peace": "https://www.gutenberg.org/ebooks/2600.txt.utf-8",
}
for filename, url in DATASOURCE.items():
    if not os.path.exists(f"{filename}.txt"):
        response = requests.get(url)
        with open(f"{filename}.txt", "wb") as f:
            f.write(response.content)

In [6]:
# Read and preprocess the text
def preprocess_gutenberg(filename):
    with open(filename, "r", encoding="utf-8") as f:
        text = f.read()

    # Find the start and end of the actual content
    start = text.find("*** START OF THE PROJECT GUTENBERG EBOOK")
    start = text.find("\n", start) + 1
    end = text.find("*** END OF THE PROJECT GUTENBERG EBOOK")

    # Extract the main content
    text = text[start:end].strip()

    # Basic preprocessing
    # Remove multiple newlines and spaces
    text = "\n".join(line.strip() for line in text.split("\n") if line.strip())
    return text

In [8]:
def get_dataset_text():
    all_text = []
    for filename in DATASOURCE:
        text = preprocess_gutenberg(f"{filename}.txt")
        all_text.append(text)
    return all_text

texts = get_dataset_text()

In [14]:
vectorizer = tf.keras.layers.TextVectorization(
	output_mode="int",
	output_sequence_length=100,
)

In [ ]:
vectorizer.adapt(texts)
len(vectorizer.get_vocabulary())

56003

In [20]:
vectorizer(texts)

<tf.Tensor: shape=(11, 100), dtype=int64, numpy=
array([[25224,    52,     2, ..., 12775,     2,  5242],
       [ 5954,    52,     2, ...,    14,    82,  1309],
       [ 4739,    32, 28155, ...,   158,  2558,   888],
       ...,
       [    6,  2133,     5, ...,     6,  1353,  1540],
       [    2,  3481,     5, ...,     9,    29,  1618],
       [  617,     3,   802, ...,   158,  3428,   158]], shape=(11, 100))>

In [23]:
def pos_enc_matrix(L, d, n=10000):
    """Create positional encoding matrix

    Args:
        L: Input dimension (length)
        d: Output dimension (depth), even only
        n: Constant for the sinusoidal functions

    Returns:
        numpy matrix of floats of dimension L-by-d. At element (k,2i) the value
        is sin(k/n^(2i/d)) while at element (k,2i+1) the value is cos(k/n^(2i/d))
    """
    assert d % 2 == 0, "Output dimension needs to be an even integer"
    d2 = d//2
    P = np.zeros((L, d))
    k = np.arange(L).reshape(-1, 1)     # L-column vector
    i = np.arange(d2).reshape(1, -1)    # d-row vector
    denom = np.power(n, -i/d2)          # n**(-2*i/d)
    args = k * denom                    # (L,d) matrix
    P[:, ::2] = np.sin(args)
    P[:, 1::2] = np.cos(args)
    return P

class PositionalEmbedding(tf.keras.layers.Layer):
    """Positional embedding layer. Assume tokenized input, transform into
    embedding and returns positional-encoded output."""
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        """
        Args:
            sequence_length: Input sequence length
            vocab_size: Input vocab size, for setting up embedding matrix
            embed_dim: Embedding vector size, for setting up embedding matrix
        """
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim     # d_model in paper
        # token embedding layer: Convert integer token to D-dim float vector
        self.token_embeddings = tf.keras.layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim, mask_zero=True
        )
        # positional embedding layer: a matrix of hard-coded sine values
        matrix = pos_enc_matrix(sequence_length, embed_dim)
        self.position_embeddings = tf.constant(matrix, dtype="float32")

    def call(self, inputs):
        """Input tokens convert into embedding vectors then superimposed
        with position vectors"""
        embedded_tokens = self.token_embeddings(inputs)
        return embedded_tokens + self.position_embeddings

    # this layer is using an Embedding layer, which can take a mask
    # see https://www.tensorflow.org/guide/keras/masking_and_padding#passing_mask_tensors_directly_to_layers
    def compute_mask(self, *args, **kwargs):
        return self.token_embeddings.compute_mask(*args, **kwargs)

    def get_config(self):
        # to make save and load a model using custom layer possible
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "vocab_size": self.vocab_size,
            "embed_dim": self.embed_dim,
        })
        return config

In [ ]:
class SelfAttention(tf.keras.Layer):
	def __init__(self, vocabulary_size, num_heads):
		super().__init__()
		self.inputs = tf.keras.layers.Input(shape=input_shape, dtype='float32')
		self.transformers = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=64)
		self.layer_norm = tf.keras.layers.LayerNormalization()
		self.add = tf.keras.layers.Add()
	def call(self, inputs):
		x = self.transformers(inputs)
		x = self.layer_norm(x)
		x = self.add([x, inputs])
		return x

<tf.Tensor: shape=(11, 100, 1024), dtype=float32, numpy=
array([[[-0.0499872 , -0.03755436,  0.01187991, ...,  0.01611057,
         -0.04102062,  0.02398309],
        [ 0.03859569,  0.04232898, -0.01947398, ..., -0.02422129,
         -0.02529659, -0.01592542],
        [ 0.00811176, -0.02524625, -0.0465725 , ..., -0.01529358,
          0.00835401, -0.04109666],
        ...,
        [-0.03999171,  0.00013579,  0.00937694, ...,  0.02267036,
         -0.00497852, -0.01017249],
        [ 0.00811176, -0.02524625, -0.0465725 , ..., -0.01529358,
          0.00835401, -0.04109666],
        [-0.00179823,  0.01607274,  0.01094253, ...,  0.0457007 ,
         -0.02562829,  0.0034737 ]],

       [[-0.02880467, -0.00713871,  0.00517131, ...,  0.02952436,
          0.04795006,  0.04186695],
        [ 0.03859569,  0.04232898, -0.01947398, ..., -0.02422129,
         -0.02529659, -0.01592542],
        [ 0.00811176, -0.02524625, -0.0465725 , ..., -0.01529358,
          0.00835401, -0.04109666],
        ..